# 03 — Generics, `TypeVar` et PEP 695

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre l'intérêt des génériques
- écrire une fonction générique avec la syntaxe PEP 695 (`def f[T]`)
- écrire une classe générique avec `class Stack[T]:`
- comparer avec l'ancien style `TypeVar`
- maîtriser la notion de variance (brièvement)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- type hints modernes
- `Protocol`, `TypedDict`, `Literal`
- classes, héritage

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- métaclasses et autres sujets avancés (formation Avancé)

## Plan

1. Pourquoi des génériques
2. Fonction générique — PEP 695
3. Classe générique — PEP 695
4. Ancien style : `TypeVar` et `Generic[T]`
5. Contraintes et bornes
6. Variance (survol)
7. Synthèse
8. Exercices

---

## 1. Pourquoi des génériques

Une fonction qui accepte et renvoie « une liste de n'importe quoi » sans perdre l'information de type : c'est exactement ce que permettent les génériques.

In [ ]:
def premier_non_generique(valeurs: list[object]) -> object:
    return valeurs[0]


In [ ]:
v = premier_non_generique([1, 2, 3])
# mypy pense que v est object → pas d'autocomplétion sur les méthodes int
v + 1  # mypy : error !


---

## 2. Fonction générique — PEP 695 (3.12+)

Syntaxe moderne : `def f[T](...)`. Plus besoin d'importer `TypeVar`.

In [ ]:
def premier[T](valeurs: list[T]) -> T:
    return valeurs[0]


In [ ]:
premier([1, 2, 3])        # mypy infère int → retourne int
premier(['a', 'b'])       # infère str → retourne str


Dans un éditeur avec mypy/pyright, vous voyez que `premier([1])` a pour type `int`. C'est la magie des génériques.

---

## 3. Classe générique — PEP 695

La même syntaxe s'applique aux classes : `class Stack[T]:`.

In [ ]:
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, x: T) -> None:
        self._items.append(x)
    def pop(self) -> T:
        return self._items.pop()
    def __len__(self) -> int:
        return len(self._items)


In [ ]:
s: Stack[int] = Stack()
s.push(1); s.push(2); s.push(3)
s.pop()


In [ ]:
mots: Stack[str] = Stack()
mots.push('bonjour'); mots.pop()


---

## 4. Ancien style : `TypeVar` et `Generic[T]`

C'est ce que vous rencontrerez dans une large majorité du code existant. Python 3.12+ propose PEP 695 mais la base installée des projets existants est encore massivement en `TypeVar`.

In [ ]:
from typing import Generic, TypeVar

T = TypeVar('T')


class StackAncien(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, x: T) -> None:
        self._items.append(x)
    def pop(self) -> T:
        return self._items.pop()


In [ ]:
StackAncien[int]().__class_getitem__


---

## 5. Contraintes et bornes

Limiter les types acceptés par un générique. Deux mécanismes.

### Avec PEP 695 : `T: Base`

In [ ]:
class Comparable:
    def __lt__(self, other: object) -> bool: ...

def plus_petit[T: int | float](a: T, b: T) -> T:
    return a if a < b else b


In [ ]:
plus_petit(3, 5)


In [ ]:
plus_petit(1.5, 0.5)


### Avec l'ancien `TypeVar(bound=...)`

In [ ]:
from typing import TypeVar
from numbers import Real

TR = TypeVar('TR', bound=Real)

def plus_grand(a: TR, b: TR) -> TR:
    return a if a > b else b

plus_grand(10, 20)


---

## 6. Variance (survol)

La variance définit comment les génériques se comportent sous l'héritage.

- **Invariant (défaut)** : `Stack[Animal]` n'est pas compatible avec `Stack[Chien]`.
- **Covariant** : `Sequence[Chien]` est compatible là où `Sequence[Animal]` est attendu (lecture seule).
- **Contravariant** : typique des callbacks (`Callable[[Animal], None]` compatible avec `Callable[[Chien], None]` par contravariance sur l'argument).

C'est un sujet subtil — dans 95 % des cas pratiques, l'invariance par défaut suffit. Si vous en avez besoin, documentez-vous sur `Generic[T]` avec `covariant=True` / `contravariant=True`.

---

## Synthèse

| Syntaxe | Exemple | Version |
|---|---|---|
| `def f[T](x: T) -> T:` | Fonction générique | PEP 695 (3.12+) |
| `class Stack[T]:` | Classe générique | PEP 695 (3.12+) |
| `def f[T: X](x: T) -> T:` | Borne | PEP 695 |
| `T = TypeVar('T')` + `Generic[T]` | Ancien style | Toujours valide |


### Règles à retenir

1. **Code neuf en 3.12+ : syntaxe PEP 695.**
2. **Lire du code existant = connaître `TypeVar`.** Les deux coexistent.
3. **L'invariance est le cas par défaut.** Ne touchez à la variance que si vous savez pourquoi.
4. **Les génériques servent la lecture** : `Stack[int]` dit plus que `Stack`.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Fonction `dernier` *(facile)*

Écrire une fonction générique `dernier[T](valeurs: list[T]) -> T` qui renvoie le dernier élément.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Generics", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def dernier[T](valeurs: list[T]) -> T:
    return valeurs[-1]

print(dernier([1, 2, 3]))
print(dernier(['a', 'b', 'c']))
```

</details>

### Exercice 2 — Queue générique *(moyen)*

Écrire `class Queue[T]:` avec `enqueue`, `dequeue`, `__len__`. Utiliser une `list[T]` en interne.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Generics", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
class Queue[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def enqueue(self, x: T) -> None:
        self._items.append(x)
    def dequeue(self) -> T:
        return self._items.pop(0)
    def __len__(self) -> int:
        return len(self._items)

q: Queue[str] = Queue()
q.enqueue('a'); q.enqueue('b')
print(q.dequeue(), len(q))
```

</details>

### Exercice 3 — Paire typée *(moyen)*

Écrire `class Paire[A, B]:` avec deux attributs `a: A` et `b: B`, et une méthode `swap(self) -> Paire[B, A]`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Generics", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Paire[A, B]:
    def __init__(self, a: A, b: B) -> None:
        self.a = a
        self.b = b
    def swap(self) -> 'Paire[B, A]':
        return Paire(self.b, self.a)
    def __repr__(self) -> str:
        return f'Paire({self.a!r}, {self.b!r})'

p = Paire(1, 'un')
print(p.swap())
```

</details>

### Exercice 4 — Registre générique *(difficile)*

Écrire `class Registre[K, V]:` qui enveloppe un `dict[K, V]` avec méthodes `enregistrer(self, k: K, v: V) -> None`, `chercher(self, k: K) -> V | None`, `tout(self) -> dict[K, V]`. Utiliser le registre avec `K=str, V=int`, puis `K=int, V=str`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Generics", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class Registre[K, V]:
    def __init__(self) -> None:
        self._data: dict[K, V] = {}
    def enregistrer(self, k: K, v: V) -> None:
        self._data[k] = v
    def chercher(self, k: K) -> V | None:
        return self._data.get(k)
    def tout(self) -> dict[K, V]:
        return dict(self._data)

r1: Registre[str, int] = Registre()
r1.enregistrer('alice', 30)
r1.enregistrer('bob', 25)
print(r1.tout())

r2: Registre[int, str] = Registre()
r2.enregistrer(1, 'un'); r2.enregistrer(2, 'deux')
print(r2.chercher(1), r2.chercher(3))
```

</details>

---

## Ressources externes

### Documentation officielle
- [Generic types — `typing`](https://docs.python.org/3/library/typing.html#generics)
- [mypy generics tutorial](https://mypy.readthedocs.io/en/stable/generics.html)

### PEPs de référence
- **PEP 484** — Type Hints
- **PEP 483** — Theory of type hints
- **PEP 695** — Type parameter syntax